# 3. 정답 마스크 라벨링 (3_labeling)

`LABELING.md` 규칙대로 그릇마다 번호를 칠해서 저장합니다. **허깅페이스 토큰 필요 없음.**

* 초안: DINO + SAM이 그릇 후보를 먼저 깔아 줌 → 틀린 곳만 클릭으로 고치기
* **초안은 모델 결과라 틀릴 수 있음. 한 장씩 꼭 눈으로 확인하고 저장**
* 저장 위치: `dataset/gt_inst/train/` 또는 `dataset/gt_inst/test/` (화면에서 고름, 기본은 train). 흑백 정답 `dataset/gt/`는 자동 생성
* 파일 이름 = 원본 사진 이름

**순서:** 0 → 1 → 2번 셀 실행 → 3번 셀에서 `start('hwang')` 또는 `start('jeong')`

# 0. 설치

In [ ]:
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q pillow-heif

# 1. 경로, 설정, 모델

In [ ]:
import os, glob, json, io, base64, html
import numpy as np
import torch
import cv2
from PIL import Image, ImageOps, ImageDraw, ImageFont
from pillow_heif import register_heif_opener
from scipy import ndimage
from IPython.display import display, HTML, JSON
from google.colab import drive, output

register_heif_opener()
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AdCanvas_nukki'
IMG_DIR = f'{BASE}/dataset/images'          # 사람별 폴더: images/hwang/, images/jeong/ (train, test 구분 없이)
GT_INST_DIR = f'{BASE}/dataset/gt_inst'     # 그릇별 정답: gt_inst/train/, gt_inst/test/
GT_DIR = f'{BASE}/dataset/gt'               # 흑백 정답 (자동 생성): gt/train/, gt/test/
SPLITS = ('train', 'test')
EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif')


def stem(p):
    return os.path.splitext(os.path.basename(p))[0]


def scan_images():
    """사진 이름(확장자 뺀 원본 이름) → 경로. 사람 폴더(images/hwang 등) 안의 사진만 읽음
    두 폴더에 같은 이름이 있으면 덮어쓰기를 막으려고 목록에서 빼고 경고"""
    found = {}
    for p in sorted(glob.glob(f'{IMG_DIR}/*/*')):
        if p.lower().endswith(EXTS):
            found.setdefault(stem(p), []).append(p)
    dup = {n: ps for n, ps in found.items() if len(ps) > 1}
    loose = [p for p in glob.glob(f'{IMG_DIR}/*') if p.lower().endswith(EXTS)]
    return {n: ps[0] for n, ps in found.items() if len(ps) == 1}, dup, loose


def label_split(n):
    """이 사진의 정답이 어느 폴더(train/test)에 있는지. 없으면 None"""
    return next((s for s in SPLITS if os.path.exists(f'{GT_INST_DIR}/{s}/{n}.png')), None)


IMAGES, DUP, LOOSE = scan_images()
for s in SPLITS:
    os.makedirs(f'{GT_INST_DIR}/{s}', exist_ok=True)
    os.makedirs(f'{GT_DIR}/{s}', exist_ok=True)
if DUP:
    print('경고: 두 폴더에 이름이 같은 사진이 있어서 뺐어요. 한쪽 이름 끝에 _2를 붙여 주세요:')
    for n, ps in DUP.items():
        print('  -', n, ':', [os.path.relpath(p, IMG_DIR) for p in ps])
if LOOSE:
    print(f'참고: images/ 바로 아래 사진 {len(LOOSE)}장은 읽지 않아요. images/hwang/ 또는 images/jeong/으로 옮겨 주세요.')

def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


device = get_device()

def load_image(p, max_side=2048):
    img = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
    if max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    return img

LAB = dict(
    disp_side=1100,           # 화면에 보여 주는 크기 (긴 변)
    sam_model='facebook/sam2.1-hiera-small',
    utensil_words=['spoon', 'chopsticks'],
    utensil_overlap=0.2,      # 수저의 20% 이상이 그릇 외곽 안에 있으면 그 그릇에 걸친 것
    inside_drop=0.9,          # 초안에서 다른 그릇 안에 90% 이상 들어간 후보는 중복으로 뺌
    box_margin=0.05,          # 박스 밖에 포함 점을 찍으면 박스를 넓힘
    goal={'train': 100, 'test': 20},
)

from sam2.sam2_image_predictor import SAM2ImagePredictor
predictor = SAM2ImagePredictor.from_pretrained(LAB['sam_model'], device=device)

from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from torchvision.ops import nms

GD_ID = 'IDEA-Research/grounding-dino-tiny'   # 공개 모델, 허깅페이스 토큰 필요 없음
gd_proc = AutoProcessor.from_pretrained(GD_ID)
gd = AutoModelForZeroShotObjectDetection.from_pretrained(GD_ID).to(device).eval()

DINO_CFG = dict(
    dish_words=['plate', 'bowl', 'dish', 'pot'],            # 그릇 (기본 선택)
    other_words=['cup', 'glass', 'spoon', 'chopsticks'],    # 후보로만 표시 (기본 선택 해제)
    food_words=['food'],                                    # 그릇 없이 놓인 음식
    box_th=0.3,             # 단어별 확신도 중 가장 높은 값의 기준
    nms_iou=0.6,            # 같은 물체에 겹친 박스 정리
    max_area=0.9,           # 사진의 90% 넘는 박스는 버림 (테이블, 배경)
    food_max_area=0.5,      # food 박스는 사진의 50%까지만
)


def box_area(b):
    return (b[2] - b[0]) * (b[3] - b[1])


@torch.no_grad()
def dino_boxes(img):
    """사진 → [{'box', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택)}]"""
    C = DINO_CFG
    words = C['dish_words'] + C['other_words'] + C['food_words']
    kinds = ['dish'] * len(C['dish_words']) + ['other'] * len(C['other_words']) + ['food'] * len(C['food_words'])
    inputs = gd_proc(images=img, text='. '.join(words) + '.', return_tensors='pt').to(device)
    out = gd(**inputs)
    probs = out.logits[0].sigmoid().float().cpu()
    ids = inputs['input_ids'][0].tolist()
    cols = []
    for w in words:
        sub = gd_proc.tokenizer(w, add_special_tokens=False)['input_ids']
        pos = [i + j for i in range(len(ids) - len(sub) + 1) if ids[i:i + len(sub)] == sub for j in range(len(sub))]
        cols.append(probs[:, pos].max(1).values if pos else torch.zeros(len(probs)))
    score, best = torch.stack(cols, 1).max(1)
    W, H = img.size
    cx, cy, bw, bh = out.pred_boxes[0].float().cpu().unbind(-1)
    boxes = torch.stack([(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H], -1)
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, W)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, H)
    keep = [i for i in range(len(boxes)) if score[i] > C['box_th'] and box_area(boxes[i].tolist()) <
            (C['food_max_area'] if kinds[int(best[i])] == 'food' else C['max_area']) * W * H]
    if not keep:
        return []
    keep = torch.tensor(keep)
    k = keep[nms(boxes[keep], score[keep], C['nms_iou'])].tolist()
    return [{'box': boxes[i].tolist(), 'label': words[int(best[i])], 'score': float(score[i]),
             'kind': kinds[int(best[i])], 'dish': kinds[int(best[i])] != 'other'} for i in k]

done = {s: len(glob.glob(f'{GT_INST_DIR}/{s}/*.png')) for s in SPLITS}
people = {d: len([n for n, p in IMAGES.items() if os.path.basename(os.path.dirname(p)) == d])
          for d in sorted({os.path.basename(os.path.dirname(p)) for p in IMAGES.values()})}
print(f"device: {device} | 사진 {people} | 저장된 정답 train {done['train']}/{LAB['goal']['train']}, test {done['test']}/{LAB['goal']['test']}")

# 2. 라벨링 도구
실행만 하면 됩니다 (화면은 3번 셀에서 열림).

| 하고 싶은 것 | 방법 |
|---|---|
| 그릇 고르기 | 아래 번호 버튼, 또는 `선택` 모드에서 그릇 클릭 |
| 포함 / 제외 | `포함 점` 모드에서 클릭 / **Shift+클릭 = 제외** |
| 새 그릇, 새 수저 | `박스로 새 그릇` / `박스로 새 수저` → 두 꼭짓점 차례로 클릭 |
| 수저의 주인 | 수저를 고른 뒤 `주인:` 옆 번호 |
| 음식이 비어 있음 | `구멍 채우기` |
| 학습용 / 시험용 | 위쪽 `train` / `test` 버튼 (저장할 폴더) |
| 저장 | `저장하고 다음` (저장 안 하고 넘어가면 수정 내용이 사라짐) |

화면의 번호가 저장되는 번호예요 (큰 그릇부터, 수저는 맨 뒤).

In [ ]:
def mask_box(m):
    ys, xs = np.nonzero(m)
    return [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def box_mask(shape, box, expand=0.1):
    x0, y0, x1, y1 = box
    ex, ey = expand * (x1 - x0), expand * (y1 - y0)
    out = np.zeros(shape, bool)
    out[max(int(y0 - ey), 0):max(int(np.ceil(y1 + ey)), 0), max(int(x0 - ex), 0):max(int(np.ceil(x1 + ex)), 0)] = True
    return out


def local(fn, m, pad=0):
    """마스크가 있는 영역만 잘라서 계산 (사진 전체로 계산하면 매우 느림)"""
    out = np.zeros_like(m)
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return out
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1])
    return out


def _hull_crop(c):
    pts = cv2.findNonZero(c.astype(np.uint8))
    out = np.zeros(c.shape, np.uint8)
    cv2.fillConvexPoly(out, cv2.convexHull(pts), 1)
    return out.astype(bool)


def hull(m):
    """볼록 외곽 (그릇을 고무줄로 감싼 영역)"""
    return local(_hull_crop, m)


def close_fill(m):
    """틈 메우기 + 구멍 채우기"""
    return local(lambda c: ndimage.binary_fill_holes(ndimage.binary_closing(c, iterations=3)), m, pad=4)


def pick_whole(ms, ss, box, mm_score=0.6, mm_inside=0.9):
    """SAM 후보 마스크 3개(부분 ~ 전체) 중 점수가 충분하고 박스 안에 들어오는 것 중 가장 큰 것 = 전체 물체"""
    ok = [j for j in range(len(ms)) if ms[j].any() and ss[j] >= mm_score * ss.max()]
    bm = box_mask(ms[0].shape, box)
    ok = [j for j in ok if (ms[j] & bm).sum() >= mm_inside * ms[j].sum()]
    return ms[max(ok, key=lambda j: ms[j].sum())] if ok else ms[int(ss.argmax())]


COLORS = np.array([(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180),
                   (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128),
                   (220, 190, 255), (170, 110, 40), (128, 0, 0), (170, 255, 195), (128, 128, 0)])


class Labeler:
    def __init__(self, names):
        self.names, self.uid = names, 0
        self.load(0)

    # ---------- 사진 불러오기 ----------
    def load(self, i):
        self.i = i % len(self.names)
        self.name = self.names[self.i]
        self.img = load_image(IMAGES[self.name])
        self.np = np.array(self.img)
        self.H, self.W = self.np.shape[:2]
        self.ds = min(LAB['disp_side'] / max(self.W, self.H), 1.0)
        self.disp_size = (round(self.W * self.ds), round(self.H * self.ds))
        self.disp_base = self.img.resize(self.disp_size, Image.BILINEAR)
        predictor.set_image(self.np)
        self.mode, self.corner, self.msg, self.dirty = 'point', None, '', False
        saved = label_split(self.name)
        self.saved_split, self.split = saved, saved or 'train'
        self.objs = self.from_saved() if saved else self.prelabel()
        self.cur = self.ordered()[0]['uid'] if self.objs else None

    def new_obj(self, kind, box, mask=None):
        self.uid += 1
        box = [max(0.0, box[0]), max(0.0, box[1]), min(float(self.W), box[2]), min(float(self.H), box[3])]
        if mask is None:
            m, s, _ = predictor.predict(box=np.array(box, np.float32), multimask_output=True)
            m, s = np.asarray(m) > 0.5, np.asarray(s)
            mask = close_fill(pick_whole(m, s, box)) if kind == 'dish' else m[int(s.argmax())]
        return {'uid': self.uid, 'type': kind, 'parent': None, 'box': box,
                'points': [], 'labels': [], 'base': mask, 'mask': mask}

    def prelabel(self):
        found = dino_boxes(self.img)
        dishes = [self.new_obj('dish', f['box']) for f in found if f['dish']]
        dishes = sorted([d for d in dishes if d['mask'].any()], key=lambda d: -d['mask'].sum())
        objs = []
        for d in dishes:   # 큰 그릇 안에 거의 다 들어간 후보(그릇 위 음식 등)는 중복으로 뺌
            if not any((d['mask'] & o['mask']).sum() >= LAB['inside_drop'] * d['mask'].sum() for o in objs):
                objs.append(d)
        for f in found:
            if f['label'] in LAB['utensil_words']:
                u = self.new_obj('utensil', f['box'])
                u['parent'] = self.guess_parent(u, objs)
                if u['parent'] is not None and u['mask'].any():   # 그릇에 안 걸친 수저는 라벨 대상 아님
                    objs.append(u)
        if objs:
            self.msg = f'초안: 그릇 {sum(o["type"] == "dish" for o in objs)}개, 수저 {sum(o["type"] == "utensil" for o in objs)}개. 확인하고 고친 뒤 저장하세요.'
        else:
            self.msg = '그릇을 못 찾았어요. 박스로 직접 추가하세요.'
        return objs

    def from_saved(self):
        d = f'{GT_INST_DIR}/{self.saved_split}'
        arr = np.asarray(Image.open(f'{d}/{self.name}.png').resize((self.W, self.H), Image.NEAREST))
        meta = json.load(open(f'{d}/{self.name}.json'))
        idmap, pairs = {}, []
        for o in meta['objects']:
            m = arr == int(o['id'])
            if m.any():
                ob = self.new_obj(o['type'], mask_box(m), mask=m)
                idmap[int(o['id'])] = ob['uid']
                pairs.append((o, ob))
        for o, ob in pairs:
            if o['type'] == 'utensil':
                ob['parent'] = idmap.get(int(o.get('parent', -1)))
        self.msg = f'저장된 라벨을 불러왔어요 ({self.saved_split}, 수정 모드).'
        return [ob for _, ob in pairs]

    def guess_parent(self, u, objs):
        best, score = None, 0
        for d in objs:
            if d['type'] != 'dish' or not d['mask'].any():
                continue
            ov = (u['mask'] & hull(d['mask'])).sum()
            if ov > score:
                best, score = d['uid'], ov
        return best if score > LAB['utensil_overlap'] * max(u['mask'].sum(), 1) else None

    # ---------- 편집 ----------
    def get(self, uid):
        return next((o for o in self.objs if o['uid'] == uid), None)

    def ordered(self):
        dishes = sorted([o for o in self.objs if o['type'] == 'dish'], key=lambda o: -o['mask'].sum())
        return dishes + [o for o in self.objs if o['type'] == 'utensil']

    def refine(self, o):
        if not o['points']:
            o['mask'] = o['base']
            return
        m, _, _ = predictor.predict(point_coords=np.array(o['points'], np.float32),
                                    point_labels=np.array(o['labels'], np.int32),
                                    box=np.array(o['box'], np.float32), multimask_output=False)
        o['mask'] = np.asarray(m)[0] > 0.5

    def click(self, x, y, shift):
        X, Y = min(max(x / self.ds, 0), self.W - 1), min(max(y / self.ds, 0), self.H - 1)
        if self.mode in ('box_dish', 'box_utensil'):
            if self.corner is None:
                self.corner, self.msg = (X, Y), '반대쪽 꼭짓점을 클릭하세요.'
                return
            (a, b), self.corner = self.corner, None
            box = [min(a, X), min(b, Y), max(a, X), max(b, Y)]
            if box[2] - box[0] < 5 or box[3] - box[1] < 5:
                self.msg = '박스가 너무 작아요. 다시 그려 주세요.'
                return
            kind = 'dish' if self.mode == 'box_dish' else 'utensil'
            o = self.new_obj(kind, box)
            if kind == 'utensil':
                o['parent'] = self.guess_parent(o, self.objs)
            self.objs.append(o)
            self.cur, self.mode, self.dirty = o['uid'], 'point', True
            self.msg = f'{"그릇을" if kind == "dish" else "수저를"} 추가했어요. 포함/제외 점으로 다듬으세요.'
            if kind == 'utensil' and o['parent'] is None:
                self.msg += ' 주인 그릇을 직접 골라 주세요.'
            return
        if self.mode == 'select':
            hit = [o for o in self.ordered() if o['mask'][int(Y), int(X)]]
            if hit:
                self.cur, self.msg = hit[-1]['uid'], ''
            return
        o = self.get(self.cur)
        if o is None:
            self.msg = '먼저 그릇을 고르거나 박스로 추가하세요.'
            return
        lab = 0 if (shift or self.mode == 'neg') else 1
        o['points'].append([X, Y]); o['labels'].append(lab)
        if lab == 1:   # 박스 밖에 포함 점을 찍으면 박스를 넓힘
            bx = o['box']; mw, mh = LAB['box_margin'] * (bx[2] - bx[0]), LAB['box_margin'] * (bx[3] - bx[1])
            o['box'] = [max(0.0, min(bx[0], X - mw)), max(0.0, min(bx[1], Y - mh)),
                        min(float(self.W), max(bx[2], X + mw)), min(float(self.H), max(bx[3], Y + mh))]
        self.refine(o)
        self.dirty, self.msg = True, ''

    def act(self, a, args):
        o = self.get(self.cur)
        if a == 'click':
            self.click(*args)
        elif a == 'mode':
            self.mode, self.corner = args[0], None
            self.msg = {'box_dish': '그릇의 두 꼭짓점을 차례로 클릭하세요.', 'box_utensil': '수저의 두 꼭짓점을 차례로 클릭하세요.',
                        'select': '고를 그릇을 클릭하세요.'}.get(args[0], '')
        elif a == 'split':
            if args[0] != self.split:
                self.split, self.dirty = args[0], True
                self.msg = f'저장할 곳: {args[0]}. 저장해야 반영돼요.'
        elif a == 'select':
            self.cur, self.msg = args[0], ''
        elif a == 'parent' and o and o['type'] == 'utensil':
            o['parent'], self.dirty = args[0], True
        elif a == 'fill' and o:
            o['mask'] = o['base'] = close_fill(o['mask']); o['points'], o['labels'] = [], []
            self.dirty = True
        elif a == 'undo' and o and o['points']:
            o['points'].pop(); o['labels'].pop(); self.refine(o); self.dirty = True
        elif a == 'clear' and o:
            o['points'], o['labels'] = [], []; self.refine(o); self.dirty = True
        elif a == 'retype' and o:
            o['type'] = 'utensil' if o['type'] == 'dish' else 'dish'
            o['parent'] = self.guess_parent(o, self.objs) if o['type'] == 'utensil' else None
            for u in self.objs:
                if u['parent'] == o['uid'] and o['type'] == 'utensil':
                    u['parent'] = None
            self.dirty = True
        elif a == 'delete' and o:
            self.objs.remove(o)
            for u in self.objs:
                if u['parent'] == o['uid']:
                    u['parent'] = None
            self.cur = self.ordered()[0]['uid'] if self.objs else None
            self.dirty = True
        elif a == 'save':
            self.save()
        elif a == 'save_next':
            if self.save():
                self.load(self.i + 1)
        elif a == 'next':
            self.load(self.i + 1)
        elif a == 'prev':
            self.load(self.i - 1)
        elif a == 'redo_prelabel':
            self.objs = self.prelabel()
            self.cur = self.ordered()[0]['uid'] if self.objs else None
            self.dirty = True

    def save(self):
        order = [o for o in self.ordered() if o['mask'].any()]
        dish_uids = {o['uid'] for o in order if o['type'] == 'dish'}
        if any(o['type'] == 'utensil' and o['parent'] not in dish_uids for o in order):
            self.msg = '저장 안 됨: 주인 그릇이 없는 수저가 있어요 (빨간 글씨). 주인을 고르거나 삭제하세요.'
            return False
        arr = np.zeros((self.H, self.W), np.uint8)
        idmap = {}
        for k, o in enumerate(order, 1):
            arr[o['mask']] = k
            idmap[o['uid']] = k
        objects = []
        for o in order:
            k = idmap[o['uid']]
            if not (arr == k).any():
                continue   # 다른 물체에 완전히 가려짐
            ob = {'id': k, 'type': o['type']}
            if o['type'] == 'utensil':
                ob['parent'] = idmap[o['parent']]
            objects.append(ob)
        present = {ob['id'] for ob in objects}
        if any(ob['type'] == 'utensil' and ob['parent'] not in present for ob in objects):
            self.msg = '저장 안 됨: 수저의 주인 그릇이 다른 물체에 완전히 가려졌어요.'
            return False
        for s in SPLITS:   # 다른 폴더(train/test)에 예전 저장본이 있으면 지움 (옮기기)
            if s != self.split:
                for p in (f'{GT_INST_DIR}/{s}/{self.name}.png', f'{GT_INST_DIR}/{s}/{self.name}.json', f'{GT_DIR}/{s}/{self.name}.png'):
                    if os.path.exists(p):
                        os.remove(p)
        Image.fromarray(arr).save(f'{GT_INST_DIR}/{self.split}/{self.name}.png')
        json.dump({'objects': objects, 'note': ''}, open(f'{GT_INST_DIR}/{self.split}/{self.name}.json', 'w'),
                  ensure_ascii=False, indent=1)
        Image.fromarray(((arr > 0) * 255).astype(np.uint8)).save(f'{GT_DIR}/{self.split}/{self.name}.png')
        self.saved_split, self.dirty = self.split, False
        self.msg = f'{self.split}에 저장했어요: 그릇 {sum(o["type"] == "dish" for o in objects)}개, 수저 {sum(o["type"] == "utensil" for o in objects)}개'
        return True

    # ---------- 화면 ----------
    def render(self):
        img = np.array(self.disp_base).astype(np.float32)
        order = self.ordered()
        dm = {o['uid']: np.asarray(Image.fromarray(o['mask']).resize(self.disp_size, Image.NEAREST)) for o in order}
        for k, o in enumerate(order):
            m = dm[o['uid']]
            img[m] = img[m] * 0.5 + COLORS[k % len(COLORS)] * 0.5
        out = Image.fromarray(img.clip(0, 255).astype(np.uint8))
        try:
            font = ImageFont.load_default(size=18)
        except TypeError:
            font = ImageFont.load_default()
        cur = self.get(self.cur)
        if cur is not None and cur['uid'] in dm:
            m = dm[cur['uid']]
            arr = np.array(out); arr[m ^ ndimage.binary_erosion(m, iterations=2)] = (255, 255, 0); out = Image.fromarray(arr)
        d = ImageDraw.Draw(out)
        if cur is not None:
            x0, y0, x1, y1 = [v * self.ds for v in cur['box']]
            d.rectangle([x0, y0, x1, y1], outline=(255, 255, 0), width=1)
            for (px, py), lb in zip(cur['points'], cur['labels']):
                px, py = px * self.ds, py * self.ds
                d.ellipse([px - 6, py - 6, px + 6, py + 6], fill=(0, 220, 0) if lb else (230, 0, 0), outline=(255, 255, 255), width=2)
        for k, o in enumerate(order, 1):
            m = dm[o['uid']]
            if m.any():
                ys, xs = np.nonzero(m)
                d.text((float(np.median(xs)), float(np.median(ys))), f'{k}' if o['type'] == 'dish' else f'{k}s',
                       fill=(255, 255, 255), font=font, stroke_width=3, stroke_fill=(0, 0, 0))
        if self.corner is not None:
            cx, cy = self.corner[0] * self.ds, self.corner[1] * self.ds
            d.line([cx - 10, cy, cx + 10, cy], fill=(255, 255, 0), width=3)
            d.line([cx, cy - 10, cx, cy + 10], fill=(255, 255, 0), width=3)
        buf = io.BytesIO()
        out.save(buf, 'JPEG', quality=85)
        return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()

    def info(self):
        def btn(label, a, args=(), on=False, color=None):
            style = 'margin:2px;padding:4px 9px;border-radius:6px;border:1px solid #888;cursor:pointer;'
            style += 'font-weight:bold;border:2px solid #111;' if on else ''
            style += f'background:{color};' if color else ''
            return f'<button style="{style}" onclick=\'labAct("{a}", {json.dumps(list(args))})\'>{html.escape(label)}</button>'
        order = self.ordered()
        num = {o['uid']: k for k, o in enumerate(order, 1)}
        cnt = {s: len(glob.glob(f'{GT_INST_DIR}/{s}/*.png')) for s in SPLITS}
        mine = sum(label_split(n) is not None for n in self.names)
        state = (f'저장됨 ({self.saved_split})' if self.saved_split and not self.dirty
                 else '수정 중 (저장 안 됨)' if self.dirty else '아직 저장 안 됨')
        h = [f'<div style="font-size:16px;margin:4px 0"><b>{html.escape(self.name)}</b> ({self.i + 1}/{len(self.names)}, 이 목록 완료 {mine}) | <b>{state}</b>'
             f' | 전체 train {cnt["train"]}/{LAB["goal"]["train"]}, test {cnt["test"]}/{LAB["goal"]["test"]}</div>']
        h.append('<div>저장할 곳: ' + btn('train (학습)', 'split', ['train'], self.split == 'train', '#e3f2fd')
                 + btn('test (시험)', 'split', ['test'], self.split == 'test', '#fff3e0') + '</div>')
        h.append('<div>모드: ' + ''.join(btn(l, 'mode', [m], self.mode == m) for l, m in
                 [('포함 점', 'point'), ('제외 점', 'neg'), ('선택', 'select'), ('박스로 새 그릇', 'box_dish'), ('박스로 새 수저', 'box_utensil')])
                 + ' <span style="color:#666">Shift+클릭 = 제외 점</span></div>')
        chips = []
        for k, o in enumerate(order):
            c = 'rgb(%d,%d,%d)' % tuple(COLORS[k % len(COLORS)])
            lab = f'{k + 1} 그릇' if o['type'] == 'dish' else f'{k + 1} 수저→{num.get(o["parent"], "?")}'
            chips.append(btn(lab, 'select', [o['uid']], o['uid'] == self.cur, c))
        h.append('<div>물체: ' + (''.join(chips) if chips else '<span style="color:#666">없음</span>') + '</div>')
        o = self.get(self.cur)
        if o is not None:
            row = '선택한 물체: ' + btn('구멍 채우기', 'fill') + btn('점 하나 취소', 'undo') + btn('점 모두 지우기', 'clear') \
                + btn('그릇↔수저', 'retype') + btn('삭제', 'delete', color='#f4c7c3')
            if o['type'] == 'utensil':
                ok = o['parent'] in num and self.get(o['parent'])['type'] == 'dish'
                row += ' | 주인: ' + ''.join(btn(str(num[d['uid']]), 'parent', [d['uid']], d['uid'] == o['parent'])
                                            for d in order if d['type'] == 'dish') + ('' if ok else ' <b style="color:#c00">주인 없음</b>')
            h.append(f'<div>{row}</div>')
        h.append('<div style="margin-top:4px">' + btn('이전', 'prev') + btn('저장', 'save', color='#c8e6c9')
                 + btn('저장하고 다음', 'save_next', color='#a5d6a7') + btn('다음 (저장 안 함)', 'next')
                 + btn('초안 다시 만들기', 'redo_prelabel') + '</div>')
        if self.msg:
            h.append(f'<div style="margin-top:4px;color:#0b57d0">{html.escape(self.msg)}</div>')
        return ''.join(h)

    def payload(self):
        return {'img': self.render(), 'info': self.info()}


def _act(a, args):
    try:
        lab.act(a, list(args))
    except Exception as e:
        lab.msg = f'에러: {type(e).__name__}: {e}'
    return JSON(lab.payload())


output.register_callback('lab.act', _act)

PAGE = """
<div id="labinfo" style="font-family:sans-serif;font-size:14px">__INFO__</div>
<img id="labimg" src="__IMG__" style="max-width:100%;cursor:crosshair;margin-top:6px" onclick="labClick(event, this)">
<script>
async function labAct(a, args) {
  document.getElementById('labinfo').style.opacity = 0.5;
  const r = await google.colab.kernel.invokeFunction('lab.act', [a, args], {});
  const d = r.data['application/json'];
  document.getElementById('labimg').src = d.img;
  document.getElementById('labinfo').innerHTML = d.info;
  document.getElementById('labinfo').style.opacity = 1;
}
function labClick(e, el) {
  const r = el.getBoundingClientRect();
  labAct('click', [(e.clientX - r.left) * el.naturalWidth / r.width, (e.clientY - r.top) * el.naturalHeight / r.height, e.shiftKey]);
}
</script>
"""


def start(person=None, only=None, include_done=False):
    """person: 'hwang' 또는 'jeong' (그 사람 폴더 사진만) / only: 이 사진들만 / include_done=True: 저장한 사진도 (검수, 수정용)"""
    global lab
    names = sorted(IMAGES)
    if person:
        names = [n for n in names if os.path.basename(os.path.dirname(IMAGES[n])) == person]
    if only:
        names = [n for n in names if n in set(only)]
    elif not include_done:
        names = [n for n in names if label_split(n) is None]
    if not names:
        print('라벨링할 사진이 없어요.')
        return
    lab = Labeler(names)
    p = lab.payload()
    display(HTML(PAGE.replace('__INFO__', p['info']).replace('__IMG__', p['img'])))

# 3. 라벨링 시작
* 내 사진: `start('hwang')` / `start('jeong')` (아직 라벨 안 한 사진만 차례로)
* 검수 (서로 바꿔서 보기): `start('jeong', include_done=True)`
* 특정 사진만: `start(only=['IMG_2184'])`
* 화면이 멈추거나 이상하면 이 셀을 다시 실행

In [ ]:
start('hwang')

# 3-1. test용 고르기: 색칠된 정답 모아 보기
- 3번 노트북의 0~2번 셀을 실행한 상태에서 실행
- 'name' = 사진 이름(번호)순, 'dish' = 그릇 많은 순
- WHO의 이름 수정

In [ ]:
# test 고르기: 내 정답을 색칠해서 보기
import matplotlib.pyplot as plt

WHO = 'hwang'
PER_PAGE = 12


def overlay(n, side=480):
    img = load_image(IMAGES[n], side)
    s = label_split(n)
    arr = np.asarray(Image.open(f'{GT_INST_DIR}/{s}/{n}.png').resize(img.size, Image.NEAREST))
    meta = json.load(open(f'{GT_INST_DIR}/{s}/{n}.json'))['objects']
    out = np.array(img).astype(np.float32)
    for k, o in enumerate(meta):
        m = arr == o['id']
        out[m] = out[m] * 0.45 + COLORS[k % len(COLORS)] * 0.55
    return out.clip(0, 255).astype(np.uint8), arr, meta


def counts(n):
    objs = json.load(open(f'{GT_INST_DIR}/{label_split(n)}/{n}.json'))['objects']
    return sum(o['type'] == 'dish' for o in objs), sum(o['type'] == 'utensil' for o in objs)


def my_list(sort='name'):
    """sort: 'name' = 사진 이름(번호)순, 'dish' = 그릇 많은 순"""
    names = [n for n in IMAGES if os.path.basename(os.path.dirname(IMAGES[n])) == WHO and label_split(n)]
    if sort == 'dish':
        c = {n: counts(n) for n in names}
        return sorted(names, key=lambda n: (-c[n][0], -c[n][1], n))
    return sorted(names)


def draw(names):
    rows = (len(names) + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=(15, 4.6 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, n in zip(axes.ravel(), names):
        im, arr, meta = overlay(n)
        ax.imshow(im)
        for o in meta:
            ys, xs = np.nonzero(arr == o['id'])
            if len(xs):
                ax.text(np.median(xs), np.median(ys), f"{o['id']}{'s' if o['type'] == 'utensil' else ''}",
                        color='w', fontsize=9, ha='center', va='center',
                        bbox=dict(facecolor='k', alpha=0.5, pad=1, lw=0))
        d, u = counts(n)
        ax.set_title(f"{n}\ndish {d} / utensil {u} / {label_split(n)}", fontsize=9,
                     color='tab:red' if label_split(n) == 'test' else 'k')
    plt.tight_layout(); plt.show()


def page(p=0, sort='name'):
    """page(번호, 'name' 또는 'dish')"""
    names = my_list(sort)
    part = names[p * PER_PAGE:(p + 1) * PER_PAGE]
    if not part:
        print('끝'); return
    draw(part)
    total = (len(names) + PER_PAGE - 1) // PER_PAGE
    print(f"{WHO} {len(names)}장 | test {sum(label_split(n) == 'test' for n in names)}장 | "
          f"{p + 1}/{total} 페이지 ({'이름순' if sort == 'name' else '그릇 많은 순'}) → 다음은 page({p + 1}, '{sort}')")


def show(names):
    """고른 사진만 모아 보기"""
    names = [n for n in names if label_split(n)]
    for i in range(0, len(names), PER_PAGE):
        draw(names[i:i + PER_PAGE])


page(6, 'name')

In [ ]:
# 고른 사진을 test로 옮기기 (PICK에 이름을 넣고 실행. 되돌릴 땐 TO = 'train')
import shutil
PICK = ['IMG_0834', 'IMG_2904', 'IMG_4199', 'IMG_4826', 'IMG_4920',
        'IMG_5068', 'IMG_5286', 'IMG_7509', 'IMG_7953', 'IMG_9326']
TO = 'test'
for n in PICK:
    s = label_split(n)
    if s is None: print(n, ': 정답 없음 (이름 확인)'); continue
    if s == TO: print(n, ': 이미', TO); continue
    for d, exts in ((GT_INST_DIR, ('.png', '.json')), (GT_DIR, ('.png',))):
        for ext in exts:
            if os.path.exists(f'{d}/{s}/{n}{ext}'): shutil.move(f'{d}/{s}/{n}{ext}', f'{d}/{TO}/{n}{ext}')
    print(n, ':', s, '→', label_split(n))
print('전체 train', len(glob.glob(f'{GT_INST_DIR}/train/*.png')), '/ test', len(glob.glob(f'{GT_INST_DIR}/test/*.png')))

# 4. 라벨링 현황

In [ ]:
names = sorted(IMAGES)
for who in sorted({os.path.basename(os.path.dirname(p)) for p in IMAGES.values()}):
    mine = [n for n in names if os.path.basename(os.path.dirname(IMAGES[n])) == who]
    sp = {s: [n for n in mine if label_split(n) == s] for s in SPLITS}
    print(f"{who}: 사진 {len(mine)} | train {len(sp['train'])} | test {len(sp['test'])} | 남음 {len(mine) - len(sp['train']) - len(sp['test'])}")
cnt = {s: len(glob.glob(f'{GT_INST_DIR}/{s}/*.png')) for s in SPLITS}
print(f"전체: train {cnt['train']}/{LAB['goal']['train']}, test {cnt['test']}/{LAB['goal']['test']}")
orphan = [stem(p) for s in SPLITS for p in glob.glob(f'{GT_INST_DIR}/{s}/*.png') if stem(p) not in IMAGES]
if orphan:
    print('사진이 없는 정답 (사진 이름이 바뀌었거나 지워짐):', orphan)